# Notebook 07 -- TCGA-KIRC Survival Analysis

Validates the TLS immunogenic/tolerogenic signatures (learned spatially by the GNN)
as prognostic markers in an independent bulk RNA-seq cohort.

**Rationale:** The GNN was trained on spatial Visium data and cannot be applied
directly to bulk RNA-seq. Instead, we derive a **bulk TLS functional score** by
computing per-patient z-scores for the same immunogenic and tolerogenic gene modules
used as GNN node features, then testing their prognostic value in TCGA-KIRC (n≈530).

**Data source:** RNA-seq gene expression (STAR-TPM, log2(TPM+1)) and OS data
obtained from the UCSC Xena browser (Goldman et al. 2020, Nat Biotechnol).
Generated by The Cancer Genome Atlas Research Network (TCGA 2013).

**Pipeline:**
1. Map versioned Ensembl IDs → gene symbols via GENCODE v36 probe map
2. Filter to primary tumor samples (-01A), deduplicate to one per patient
3. Z-score expression across patients per gene
4. Compute mean z-score per immunogenic and tolerogenic module
5. Bulk TLS functional score = mean(tolero_z) − mean(immuno_z)
6. Define TLS-high patients (CXCL13 z-score > 0)
7. Kaplan-Meier curves: (a) all patients, (b) TLS-high subgroup
8. Univariate Cox proportional hazards

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

PROJECT_ROOT = Path(os.environ.get('TLS_PROJECT_ROOT', Path.cwd())).resolve()
DATA_DIR     = PROJECT_ROOT / 'data' / 'paper' / 'TCGA-KIRC'
OUT_DIR      = PROJECT_ROOT / 'checkpoints'
OUT_DIR.mkdir(exist_ok=True)

EXPR_FILE    = DATA_DIR / 'TCGA-KIRC.star_tpm.tsv.gz'
SURV_FILE    = DATA_DIR / 'TCGA-KIRC.survival.tsv.gz'
PROBEMAP     = DATA_DIR / 'gencode.v36.probemap.tsv'

# TLS module gene sets (same as spatial pipeline)
IMMUNO_MODULES = {
    'b_cell_core'    : ['MS4A1', 'CD19', 'CD79A', 'CD79B'],
    'plasma_output'  : ['MZB1', 'SDC1', 'IGHG1', 'IGHG2', 'IGHA1', 'JCHAIN'],
    'tls_chemokines' : ['CCL19', 'CCL21', 'LTB'],
    't_cell_zone'    : ['CD3D', 'CD3E', 'CCR7', 'SELL'],
    'tfh'            : ['CXCR5', 'PDCD1', 'ICOS', 'BTLA'],
    'hev_markers'    : ['TNFRSF9', 'GLYCAM1', 'MADCAM1'],
}
TOLERO_MODULES = {
    'tregs'       : ['FOXP3', 'IL2RA', 'CTLA4', 'IKZF2'],
    'myeloid_sup' : ['IL10', 'TGFB1', 'CD163', 'MRC1', 'ARG1'],
    'exhaustion'  : ['HAVCR2', 'TIGIT', 'LAG3', 'PDCD1', 'TOX'],
}
CXCL13_GENE = 'CXCL13'

print('Setup complete')
print(f'Data dir: {DATA_DIR}')


Setup complete
Data dir: /home/gpeng/projects/spatial_transcriptom/tls_functional_score/data/paper/TCGA-KIRC


## 1. Gene ID Mapping

In [2]:
# Load GENCODE v36 probe map (versioned Ensembl ID -> gene symbol)
probemap = pd.read_csv(PROBEMAP, sep='\t', usecols=['id', 'gene'])
# The expression file uses versioned IDs (ENSG00000156234.15)
# The probe map also uses versioned IDs -> direct match
ensembl_to_symbol = dict(zip(probemap['id'], probemap['gene']))

# Collect all genes we need
all_sig_genes = set([CXCL13_GENE])
for genes in IMMUNO_MODULES.values():
    all_sig_genes.update(genes)
for genes in TOLERO_MODULES.values():
    all_sig_genes.update(genes)

# Build reverse map: symbol -> versioned Ensembl ID
symbol_to_ensembl = {sym: eid for eid, sym in ensembl_to_symbol.items()
                     if sym in all_sig_genes}

print(f'Probe map: {len(probemap):,} entries')
print(f'Signature genes needed: {len(all_sig_genes)}')
print(f'Mapped: {len(symbol_to_ensembl)}')
missing_map = all_sig_genes - set(symbol_to_ensembl.keys())
if missing_map:
    print(f'Not in probe map: {sorted(missing_map)}')

Probe map: 60,660 entries
Signature genes needed: 38
Mapped: 38


## 2. Load Expression Data

In [3]:
import time
print('Loading expression matrix ...')
t0 = time.time()

# Only load the rows (genes) we need
sig_ensembl_ids = set(symbol_to_ensembl.values())

# Read full first row to get column names, then filter rows on load
# (60k genes x 610 samples is ~300MB uncompressed -- load selectively)
chunks = []
with pd.read_csv(EXPR_FILE, sep='\t', index_col=0, chunksize=5000, compression='gzip') as reader:
    for chunk in reader:
        keep = chunk.index.intersection(sig_ensembl_ids)
        if len(keep):
            chunks.append(chunk.loc[keep])

expr_raw = pd.concat(chunks)   # shape: (n_sig_genes, n_samples)
print(f'  Loaded {expr_raw.shape[0]} signature genes x {expr_raw.shape[1]} samples  ({time.time()-t0:.0f}s)')

# Map Ensembl row index -> gene symbol
ensembl_to_sym_local = {v: k for k, v in symbol_to_ensembl.items()}
expr_raw.index = expr_raw.index.map(ensembl_to_sym_local)
expr_raw = expr_raw[~expr_raw.index.isna()]
expr_raw.index.name = 'gene'

print(f'  Gene symbols: {sorted(expr_raw.index.tolist())}')

# Filter to primary tumor samples only (-01A/-01B aliquot codes)
# TCGA barcode: TCGA-XX-XXXX-01A-... -> position 13-15 is sample type
# 01 = primary tumor, 11 = normal tissue
tumor_cols = [c for c in expr_raw.columns if c[13:15] == '01']
expr = expr_raw[tumor_cols].copy()   # shape: (n_genes, n_tumor_samples)
print(f'  Tumor samples: {expr.shape[1]} (of {expr_raw.shape[1]} total)')

# Derive patient ID from sample barcode: first 12 chars (TCGA-XX-XXXX)
expr.columns = pd.MultiIndex.from_arrays(
    [expr.columns, [c[:12] for c in expr.columns]],
    names=['sample_id', 'patient_id']
)
# Deduplicate: if patient has multiple tumor samples, keep mean
expr = expr.T.groupby(level='patient_id').mean().T   # (n_genes, n_unique_patients)
print(f'  Unique patients: {expr.shape[1]}')
print(f'  Expression range: [{expr.values.min():.2f}, {expr.values.max():.2f}]  (log2(TPM+1))')

Loading expression matrix ...
  Loaded 38 signature genes x 610 samples  (5s)
  Gene symbols: ['ARG1', 'BTLA', 'CCL19', 'CCL21', 'CCR7', 'CD163', 'CD19', 'CD3D', 'CD3E', 'CD79A', 'CD79B', 'CTLA4', 'CXCL13', 'CXCR5', 'FOXP3', 'GLYCAM1', 'HAVCR2', 'ICOS', 'IGHA1', 'IGHG1', 'IGHG2', 'IKZF2', 'IL10', 'IL2RA', 'JCHAIN', 'LAG3', 'LTB', 'MADCAM1', 'MRC1', 'MS4A1', 'MZB1', 'PDCD1', 'SDC1', 'SELL', 'TGFB1', 'TIGIT', 'TNFRSF9', 'TOX']
  Tumor samples: 537 (of 610 total)
  Unique patients: 533
  Expression range: [0.00, 15.86]  (log2(TPM+1))


## 3. Load Survival Data

In [4]:
surv_raw = pd.read_csv(SURV_FILE, sep='\t')
print(f'Survival rows: {len(surv_raw)}')

# Filter to tumor samples and extract patient ID
surv_raw['sample_type'] = surv_raw['sample'].str[13:15]
surv_tumor = surv_raw[surv_raw['sample_type'] == '01'].copy()
surv_tumor['patient_id'] = surv_tumor['_PATIENT']

# Deduplicate by patient (keep row with max OS.time to avoid premature censoring)
surv = (
    surv_tumor
    .sort_values('OS.time', ascending=False)
    .drop_duplicates('patient_id')
    [['patient_id', 'OS.time', 'OS']]
    .rename(columns={'OS.time': 'os_days', 'OS': 'os_event'})
    .set_index('patient_id')
)
# Remove rows with missing/zero OS time
surv = surv[surv['os_days'] > 0].dropna()

print(f'Unique tumor patients (survival): {len(surv)}')
print(f'Events (deaths): {surv["os_event"].sum()} ({100*surv["os_event"].mean():.1f}%)')
print(f'OS time (days): median={surv["os_days"].median():.0f}, '
      f'range=[{surv["os_days"].min():.0f}, {surv["os_days"].max():.0f}]')

# Inner join: patients with both expression and survival
common_patients = surv.index.intersection(expr.columns)
surv = surv.loc[common_patients]
expr = expr[common_patients]
print(f'\nMatched patients (expression + survival): {len(common_patients)}')

Survival rows: 944
Unique tumor patients (survival): 533
Events (deaths): 175 (32.8%)
OS time (days): median=1186, range=[2, 4537]

Matched patients (expression + survival): 529


## 4. Module Z-Score Computation

In [5]:
# Z-score each gene across patients (expression is already log2(TPM+1))
# Use pandas subtract/divide to preserve column names (scipy apply drops them)
gene_mean = expr.mean(axis=1)
gene_std  = expr.std(axis=1).replace(0, np.nan)          # avoid div-by-zero
expr_z    = expr.sub(gene_mean, axis=0).div(gene_std, axis=0).fillna(0)

# Verify columns are still patient IDs
assert expr_z.columns[0].startswith('TCGA-'), f"Column index lost: {expr_z.columns[:3].tolist()}"

def module_zscore(module_genes, label):
    """Mean z-score across genes present in expr_z for this module."""
    present = [g for g in module_genes if g in expr_z.index]
    absent  = [g for g in module_genes if g not in expr_z.index]
    if absent:
        print(f'  {label}: missing {absent}')
    if not present:
        print(f'  WARNING: {label} has no genes -- returning 0')
        return pd.Series(0.0, index=expr_z.columns)
    return expr_z.loc[present].mean(axis=0)

# Compute per-module z-scores (Series: index=patient_id)
print('Computing module z-scores ...')
immuno_scores = {}
for name, genes in IMMUNO_MODULES.items():
    immuno_scores[f'immuno_{name}'] = module_zscore(genes, name)

tolero_scores = {}
for name, genes in TOLERO_MODULES.items():
    tolero_scores[f'tolero_{name}'] = module_zscore(genes, name)

# CXCL13 z-score for TLS-high definition
if CXCL13_GENE in expr_z.index:
    cxcl13_z = expr_z.loc[CXCL13_GENE]
else:
    print(f'WARNING: {CXCL13_GENE} not found -- TLS-high filter disabled')
    cxcl13_z = pd.Series(0.0, index=expr_z.columns)

# Build patient-level score dataframe
scores_df = pd.DataFrame({**immuno_scores, **tolero_scores})
scores_df['cxcl13_z']      = cxcl13_z
scores_df['mean_immuno_z'] = scores_df[[c for c in scores_df.columns if c.startswith('immuno_')]].mean(axis=1)
scores_df['mean_tolero_z'] = scores_df[[c for c in scores_df.columns if c.startswith('tolero_')]].mean(axis=1)

# Bulk TLS functional score: tolerogenic dominance over immunogenic
scores_df['bulk_tls_score'] = scores_df['mean_tolero_z'] - scores_df['mean_immuno_z']

# TLS-high: CXCL13 above cohort mean (z > 0)
scores_df['tls_high'] = scores_df['cxcl13_z'] > 0

# Merge with survival
df = surv.join(scores_df)

print(f'\nFinal cohort: {len(df)} patients')
n_tls_high = int(df["tls_high"].sum())
print(f'TLS-high (CXCL13 z>0): {n_tls_high} ({100*n_tls_high/len(df):.0f}%)')
print(f'\nBulk TLS functional score distribution:')
print(df['bulk_tls_score'].describe().round(3))
print(f'\nModule z-score means (all patients):')
for col in scores_df.columns:
    if col.startswith('immuno_') or col.startswith('tolero_'):
        print(f'  {col:<30}: mean={scores_df[col].mean():.3f}  sd={scores_df[col].std():.3f}')

Computing module z-scores ...

Final cohort: 529 patients
TLS-high (CXCL13 z>0): 241 (46%)

Bulk TLS functional score distribution:
count    529.000
mean       0.000
std        0.429
min       -1.974
25%       -0.262
50%        0.017
75%        0.261
max        1.475
Name: bulk_tls_score, dtype: float64

Module z-score means (all patients):
  immuno_b_cell_core            : mean=-0.000  sd=0.829
  immuno_plasma_output          : mean=0.000  sd=0.817
  immuno_tls_chemokines         : mean=-0.000  sd=0.863
  immuno_t_cell_zone            : mean=0.000  sd=0.881
  immuno_tfh                    : mean=0.000  sd=0.824
  immuno_hev_markers            : mean=0.000  sd=0.604
  tolero_tregs                  : mean=0.000  sd=0.774
  tolero_myeloid_sup            : mean=-0.000  sd=0.669
  tolero_exhaustion             : mean=0.000  sd=0.850


## 5. Kaplan-Meier Survival Analysis

In [6]:
def km_plot(ax, df_sub, score_col, label_prefix='', cutoff='median'):
    """
    Plot KM curves for high vs low score groups.
    Returns (log_rank_p, hr_direction).
    """
    if cutoff == 'median':
        cut = df_sub[score_col].median()
    else:
        cut = cutoff

    high = df_sub[df_sub[score_col] >  cut]
    low  = df_sub[df_sub[score_col] <= cut]

    results = logrank_test(
        high['os_days'], low['os_days'],
        event_observed_A=high['os_event'],
        event_observed_B=low['os_event'],
    )
    p = results.p_value

    kmf = KaplanMeierFitter()
    for group, color, label in [
        (high, '#d73027', f'{label_prefix}High (n={len(high)})'),
        (low,  '#4575b4', f'{label_prefix}Low  (n={len(low)})'),
    ]:
        kmf.fit(group['os_days'], event_observed=group['os_event'], label=label)
        kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Overall survival probability')
    p_str = f'p={p:.3e}' if p < 0.001 else f'p={p:.3f}'
    ax.text(0.97, 0.97, f'Log-rank {p_str}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.axhline(0.5, color='k', lw=0.8, ls=':', alpha=0.4)
    ax.legend(fontsize=8)
    return p


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Panel A: all patients, bulk TLS score (median split) ---
p_all = km_plot(
    axes[0], df, 'bulk_tls_score', label_prefix='TLS score ',
)
axes[0].set_title(f'A. All KIRC patients\n(bulk TLS score, median split; n={len(df)})')

# --- Panel B: TLS-high patients only (CXCL13 z>0) ---
df_tls_high = df[df['tls_high']]
if len(df_tls_high) >= 20:
    p_tls = km_plot(
        axes[1], df_tls_high, 'bulk_tls_score', label_prefix='TLS score ',
    )
    axes[1].set_title(f'B. TLS-high patients (CXCL13 z>0)\n(bulk TLS score, median split; n={len(df_tls_high)})')
else:
    axes[1].text(0.5, 0.5, f'TLS-high n={len(df_tls_high)}\n(too few for KM)',
                 ha='center', va='center', transform=axes[1].transAxes)
    p_tls = float('nan')

# --- Panel C: CXCL13-high vs low (all patients) as a sanity check ---
p_cxcl13 = km_plot(
    axes[2], df, 'cxcl13_z', label_prefix='CXCL13 z ',
)
axes[2].set_title(f'C. All KIRC: CXCL13 expression\n(median split; n={len(df)})')

plt.suptitle('TCGA-KIRC Overall Survival -- TLS Functional Score (bulk z-score method)',
             fontsize=12, y=1.01)
plt.tight_layout()
km_fig = OUT_DIR / 'km_tcga_kirc.png'
fig.savefig(km_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {km_fig.name}')
print(f'\nLog-rank p-values:')
print(f'  All patients  (bulk TLS score): p={p_all:.4f}')
print(f'  TLS-high only (bulk TLS score): p={p_tls:.4f}' if p_tls==p_tls else
      '  TLS-high only: insufficient n')
print(f'  All patients  (CXCL13 z-score): p={p_cxcl13:.4f}')

Saved: km_tcga_kirc.png

Log-rank p-values:
  All patients  (bulk TLS score): p=0.0513
  TLS-high only (bulk TLS score): p=0.3354
  All patients  (CXCL13 z-score): p=0.0137


## 6. Cox Proportional Hazards

In [7]:
from lifelines import CoxPHFitter

cox_vars = ['bulk_tls_score', 'mean_immuno_z', 'mean_tolero_z', 'cxcl13_z']
cox_df = df[['os_days', 'os_event'] + cox_vars].dropna()

print('=== Univariate Cox PH (TCGA-KIRC OS) ===')
print(f'n={len(cox_df)}, events={cox_df["os_event"].sum()}')
print()

results_rows = []
for var in cox_vars:
    cph = CoxPHFitter()
    cph.fit(cox_df[['os_days', 'os_event', var]], duration_col='os_days', event_col='os_event')
    s = cph.summary
    hr   = float(s.loc[var, 'exp(coef)'])
    lo   = float(s.loc[var, 'exp(coef) lower 95%'])
    hi   = float(s.loc[var, 'exp(coef) upper 95%'])
    p    = float(s.loc[var, 'p'])
    results_rows.append({'variable': var, 'HR': hr, 'CI_lo': lo, 'CI_hi': hi, 'p': p})
    sig = '*' if p < 0.05 else ''
    print(f'  {var:<25}  HR={hr:.3f} [{lo:.3f}-{hi:.3f}]  p={p:.4f}  {sig}')

cox_results = pd.DataFrame(results_rows)
cox_results.to_csv(OUT_DIR / 'cox_tcga_kirc.csv', index=False)
print(f'\nSaved: cox_tcga_kirc.csv')

# Forest plot
fig2, ax2 = plt.subplots(figsize=(7, 4))
y_pos = range(len(cox_results) - 1, -1, -1)
colors = ['#d73027' if r['p'] < 0.05 else '#555555' for _, r in cox_results.iterrows()]
for i, (_, row) in zip(y_pos, cox_results.iterrows()):
    ax2.plot([row['CI_lo'], row['CI_hi']], [i, i], color=colors[len(cox_results)-1-i], lw=2)
    ax2.scatter(row['HR'], i, color=colors[len(cox_results)-1-i], s=60, zorder=5)
ax2.axvline(1.0, color='k', lw=1, ls='--', alpha=0.5)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(cox_results['variable'].tolist()[::-1])
ax2.set_xlabel('Hazard Ratio (95% CI)')
ax2.set_title('Univariate Cox PH -- TCGA-KIRC OS\n(red = p<0.05)')
plt.tight_layout()
forest_fig = OUT_DIR / 'cox_forest_tcga_kirc.png'
fig2.savefig(forest_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {forest_fig.name}')

=== Univariate Cox PH (TCGA-KIRC OS) ===
n=529, events=173

  bulk_tls_score             HR=0.827 [0.576-1.189]  p=0.3059  
  mean_immuno_z              HR=1.300 [1.043-1.622]  p=0.0196  *
  mean_tolero_z              HR=1.222 [0.970-1.539]  p=0.0891  
  cxcl13_z                   HR=1.376 [1.193-1.586]  p=0.0000  *

Saved: cox_tcga_kirc.csv
Saved: cox_forest_tcga_kirc.png


## 7. Patient Score Export

In [8]:
# Export patient-level scores + survival for downstream use
export_cols = ['os_days', 'os_event', 'tls_high', 'bulk_tls_score',
               'mean_immuno_z', 'mean_tolero_z', 'cxcl13_z']
df[export_cols].to_csv(OUT_DIR / 'patient_tls_scores_tcga_kirc.csv')
print(f'Saved: patient_tls_scores_tcga_kirc.csv  ({len(df)} patients)')

Saved: patient_tls_scores_tcga_kirc.csv  (529 patients)


## 8. Summary

In [9]:
print('=' * 60)
print('=== Notebook 07 Summary ===')
print('=' * 60)
print(f'Cohort          : TCGA-KIRC (GDC, via UCSC Xena)')
print(f'Patients        : {len(df)}')
print(f'Events          : {int(df["os_event"].sum())} ({100*df["os_event"].mean():.1f}%)')
print(f'Median OS (days): {df["os_days"].median():.0f}')
print(f'TLS-high (CXCL13 z>0): {df["tls_high"].sum()} ({100*df["tls_high"].mean():.0f}%)')
print()
print('KM log-rank p-values:')
print(f'  All patients  (bulk TLS score): p={p_all:.4f}')
if p_tls == p_tls:
    print(f'  TLS-high only (bulk TLS score): p={p_tls:.4f}')
print(f'  All patients  (CXCL13):         p={p_cxcl13:.4f}')
print()
print('Cox PH (univariate):')
for _, row in cox_results.iterrows():
    sig = '*' if row['p'] < 0.05 else ''
    print(f'  {row["variable"]:<25}  HR={row["HR"]:.3f} [{row["CI_lo"]:.3f}-{row["CI_hi"]:.3f}]  p={row["p"]:.4f}  {sig}')
print()
print('Outputs:')
for fname in ['km_tcga_kirc.png', 'cox_forest_tcga_kirc.png',
              'cox_tcga_kirc.csv', 'patient_tls_scores_tcga_kirc.csv']:
    print(f'  {fname}')

=== Notebook 07 Summary ===
Cohort          : TCGA-KIRC (GDC, via UCSC Xena)
Patients        : 529
Events          : 173 (32.7%)
Median OS (days): 1191
TLS-high (CXCL13 z>0): 241 (46%)

KM log-rank p-values:
  All patients  (bulk TLS score): p=0.0513
  TLS-high only (bulk TLS score): p=0.3354
  All patients  (CXCL13):         p=0.0137

Cox PH (univariate):
  bulk_tls_score             HR=0.827 [0.576-1.189]  p=0.3059  
  mean_immuno_z              HR=1.300 [1.043-1.622]  p=0.0196  *
  mean_tolero_z              HR=1.222 [0.970-1.539]  p=0.0891  
  cxcl13_z                   HR=1.376 [1.193-1.586]  p=0.0000  *

Outputs:
  km_tcga_kirc.png
  cox_forest_tcga_kirc.png
  cox_tcga_kirc.csv
  patient_tls_scores_tcga_kirc.csv
